In [2]:
import os, json

kaggle_config = {
    "username": "Your username",
    "key": "Your key"
}

# Create .kaggle folder
os.makedirs("/root/.kaggle", exist_ok=True)

# Save kaggle.json
with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump(kaggle_config, f)

# Set permissions
os.chmod("/root/.kaggle/kaggle.json", 600)

print("kaggle.json created successfully!")

kaggle.json created successfully!


In [ ]:
# ================================
# DOWNLOAD + EXTRACT YOUR DATASET
# ================================

# Replace with your dataset path:
DATASET = "xhlulu/140k-real-and-fake-faces"

# Download
!kaggle datasets download -d $DATASET

# Unzip
import zipfile, os

zip_file = DATASET.split("/")[-1] + ".zip"
extract_path = "/content/deepfake_dataset"

with zipfile.ZipFile(zip_file, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted to:", extract_path)

# ================================
# CHECK FOLDER STRUCTURE
# ================================
for root, dirs, files in os.walk(extract_path):
    print(f"{root} | {len(files)} files")

Dataset URL: https://www.kaggle.com/datasets/xhlulu/140k-real-and-fake-faces
License(s): other
100% 3.75G/3.75G [00:35<00:00, 113MB/s]

Dataset extracted to: /content/deepfake_dataset
/content/deepfake_dataset | 3 files
/content/deepfake_dataset/real_vs_fake | 0 files
/content/deepfake_dataset/real_vs_fake/real-vs-fake | 0 files
/content/deepfake_dataset/real_vs_fake/real-vs-fake/valid | 0 files
/content/deepfake_dataset/real_vs_fake/real-vs-fake/valid/real | 10000 files
/content/deepfake_dataset/real_vs_fake/real-vs-fake/valid/fake | 10000 files
/content/deepfake_dataset/real_vs_fake/real-vs-fake/test | 0 files
/content/deepfake_dataset/real_vs_fake/real-vs-fake/test/real | 10000 files
/content/deepfake_dataset/real_vs_fake/real-vs-fake/test/fake | 10000 files
/content/deepfake_dataset/real_vs_fake/real-vs-fake/train | 0 files
/content/deepfake_dataset/real_vs_fake/real-vs-fake/train/real | 50000 files
/content/deepfake_dataset/real_vs_fake/real-vs-fake/train/fake | 50000 files


In [ ]:
import os

dataset_path = "/content/deepfake_dataset"

print(os.listdir(dataset_path))

['valid.csv', 'test.csv', 'train.csv', 'real_vs_fake']


In [ ]:
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt

In [ ]:
train_df = pd.read_csv("/content/deepfake_dataset/train.csv")
valid_df = pd.read_csv("/content/deepfake_dataset/valid.csv")
test_df = pd.read_csv("/content/deepfake_dataset/test.csv")

print(train_df.head())

   Unnamed: 0                                      original_path     id  \
0           0  /kaggle/input/flickrfaceshq-dataset-nvidia-par...  31355   
1           1  /kaggle/input/flickrfaceshq-dataset-nvidia-par...  02884   
2           2  /kaggle/input/flickrfaceshq-dataset-nvidia-par...  33988   
3           3  /kaggle/input/flickrfaceshq-dataset-nvidia-par...  53875   
4           4  /kaggle/input/flickrfaceshq-dataset-nvidia-par...  24149   

   label label_str                  path  
0      1      real  train/real/31355.jpg  
1      1      real  train/real/02884.jpg  
2      1      real  train/real/33988.jpg  
3      1      real  train/real/53875.jpg  
4      1      real  train/real/24149.jpg  


In [ ]:
print(train_df.columns)

Index(['Unnamed: 0', 'original_path', 'id', 'label', 'label_str', 'path'], dtype='object')


In [ ]:
print(valid_df.head())
print(valid_df.columns)

   Unnamed: 0                                      original_path     id  \
0           0  /kaggle/input/flickrfaceshq-dataset-nvidia-par...  20001   
1           1  /kaggle/input/flickrfaceshq-dataset-nvidia-par...  11264   
2           2  /kaggle/input/flickrfaceshq-dataset-nvidia-par...  19817   
3           3  /kaggle/input/flickrfaceshq-dataset-nvidia-par...  46851   
4           4  /kaggle/input/flickrfaceshq-dataset-nvidia-par...  10411   

   label label_str                  path  
0      1      real  valid/real/20001.jpg  
1      1      real  valid/real/11264.jpg  
2      1      real  valid/real/19817.jpg  
3      1      real  valid/real/46851.jpg  
4      1      real  valid/real/10411.jpg  
Index(['Unnamed: 0', 'original_path', 'id', 'label', 'label_str', 'path'], dtype='object')


In [ ]:
dataset_path = "/content/deepfake_dataset/real_vs_fake/real-vs-fake"

train_df['path'] = dataset_path + "/" + train_df['path']
valid_df['path'] = dataset_path + "/" + valid_df['path']
test_df['path'] = dataset_path + "/" + test_df['path']

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

valid_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col='path',
    y_col='label_str',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

valid_generator = valid_datagen.flow_from_dataframe(
    dataframe=valid_df,
    x_col='path',
    y_col='label_str',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

test_generator = valid_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col='path',
    y_col='label_str',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

Found 100000 validated image filenames belonging to 2 classes.
Found 20000 validated image filenames belonging to 2 classes.
Found 20000 validated image filenames belonging to 2 classes.


In [ ]:
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

model = Sequential([
    base_model,
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 62720)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     8,028,288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,286,401 (39.24 MB)

 Trainable params: 8,028,417 (30.63 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [1]:
history = model.fit(
    train_generator,
    validation_data=valid_generator,
    epochs=5
)

NameError: name 'model' is not defined